# Stability gap (P1 replay, tasks 1-5) and GroupNorm V_O/h
Enable Internet, T4x2 and the `github_token` secret. Attach CIFAR-100 (`cifar-100-python/{train,test,meta}`) and the P1 output(s) containing `icarl` and `icarl_full` (Gram/power10) seeds 0 and 1, noBN, superclass. Attach this notebook's own output to resume.

Each P1 job is replayed through task 5 with the same K0 on the commit that ran its tasks 1-5 (VNA: metric_trace_repair.json source_commit if present, else schedule.code_commit; baseline: its seed's VNA commit), one checkout per commit; the commit of every job is printed before launch. Old/new-class test accuracy is read at every SGD step for r = 0-10, every 5 steps to 200 in tasks 2-5, then every epoch, under no_grad with all RNG and module modes restored. Task-boundary checkpoints (end of tasks 1-4) are kept. Epoch losses and tasks.json for tasks 1-5 must equal P1 exactly; otherwise no measurement is reported.

GroupNorm (run last in each lane): baseline iCaRL on ResNet-32 with GroupNorm(8) (lr 0.05, zero-init gamma of each block's last GroupNorm) trains task 1 behind two gates (epoch-10 loss, task-1 accuracy >= 76) and then tasks 2-5; in tasks 2-5 (every step for r = 0-10, every 5 steps to 200) old/new test accuracy is read as in the replay and a VNA engine (Gram/power10) applies one full T26 application to a discarded copy of the network and logs V_O, h, L/KD before/after, s, Delta and would_accept (T26f); the real model is checked bitwise unchanged. The virtual neuron bypasses GroupNorm (a definition choice, see preregistration). Output: `outputs/vna_stability_gap/`.


In [ ]:
REPO_COMMIT = 'ccf992469a4f902990a4d90a96b81064ecaf3ba5'
SOURCE_DIGEST = None
RUN_TAG = 'vna_stability_gap'
import time
SESSION_HOURS = 11.5
SESSION_START = time.time()
VNA_DEADLINE = SESSION_START + SESSION_HOURS*3600
DISK_CAP_BYTES = 20_000_000_000

import hashlib, json, os, shutil, subprocess, sys, threading, time, zipfile
os.environ.update(TQDM_DISABLE="1", PYTHONUNBUFFERED="1")
from pathlib import Path
from kaggle_secrets import UserSecretsClient

MAIN_URL = "https://github.com/duyh80456-code/Counterfactual-Projection.git"
REPO = Path("/kaggle/working/counterfactual-projection")
RUN_ROOT = Path("/kaggle/working/vna_stability_gap")
OUTPUT = RUN_ROOT

def private_clone(url, destination, branch):
    token = UserSecretsClient().get_secret("github_token").strip()
    if not token: raise RuntimeError("Kaggle Secret github_token is unavailable")
    askpass = Path("/kaggle/working/.counterfactual_git_askpass.py")
    askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update(GITHUB_TOKEN_RUNTIME=token, GIT_ASKPASS=str(askpass),
               GIT_TERMINAL_PROMPT="0")
    try:
        if len(branch) == 40 and all(c in "0123456789abcdef" for c in branch):
            subprocess.run(["git", "clone", "--filter=blob:none", "--no-checkout",
                            url, str(destination)], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "fetch", "origin", branch], env=env, check=True)
            subprocess.run(["git", "-C", str(destination), "checkout", "--detach", branch], env=env, check=True)
        else:
            subprocess.run(["git", "clone", "--branch", branch, "--single-branch",
                            url, str(destination)], env=env, check=True)
    finally:
        askpass.unlink(missing_ok=True)

for checkout in (REPO,):
    if checkout.exists(): shutil.rmtree(checkout)
OUTPUT.mkdir(parents=True, exist_ok=True)
private_clone(MAIN_URL, REPO, REPO_COMMIT)
print("main:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

if subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip() != REPO_COMMIT:
    raise RuntimeError("Pinned repository revision mismatch")

from datetime import datetime, timezone
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Resolved code commit:", CODE_COMMIT, "pin:", REPO_COMMIT)
commit_path = OUTPUT / "code_commit.json"
sessions = json.loads(commit_path.read_text()) if commit_path.exists() else []
sessions.append({"timestamp": datetime.now(timezone.utc).isoformat(), "pin": REPO_COMMIT, "commit": CODE_COMMIT})
temporary = commit_path.with_suffix(".json.tmp")
temporary.write_text(json.dumps(sessions, indent=2) + "\n")
temporary.replace(commit_path)

if SOURCE_DIGEST:
    bundles = [p for p in Path('/kaggle/input').rglob('vna_v3_source.zip')
               if hashlib.sha256(p.read_bytes()).hexdigest() == SOURCE_DIGEST]
    if not bundles: raise FileNotFoundError('Attach vna_v3_source.zip matching SOURCE_DIGEST; this notebook uses uncommitted source')
    with zipfile.ZipFile(bundles[0]) as archive:
        for name in archive.namelist():
            target = (REPO/name).resolve()
            if not target.is_relative_to(REPO.resolve()): raise ValueError('unsafe source bundle member')
        archive.extractall(REPO)
    os.environ['VNA_SOURCE_DIGEST'] = SOURCE_DIGEST
    shutil.copy2(bundles[0], RUN_ROOT/'vna_v3_source.zip')
    (RUN_ROOT/'source_manifest.json').write_text(json.dumps(dict(code_commit=REPO_COMMIT, source_digest=SOURCE_DIGEST)))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO) + "[test,vision,diagnostics]"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-upstream.txt")], check=True)
subprocess.run(["bash", "scripts/fetch_third_party.sh"], cwd=REPO, check=True)
RUNTIME_PYTHONPATH = os.pathsep.join(filter(None, (
    str(REPO), os.environ.get("PYTHONPATH", ""))))
test_env = os.environ.copy(); test_env.update(PYTHONUNBUFFERED="1", TQDM_DISABLE="1", PYTHONPATH=RUNTIME_PYTHONPATH, REQUIRE_UPSTREAM="1", PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION="python")
import torch
if torch.cuda.device_count() != 2:
    raise RuntimeError(f"Select T4 x2; found {torch.cuda.device_count()} GPU(s)")
required_cifar_files = {"train", "test", "meta"}
cifar_dirs = []
for directory, _, filenames in os.walk("/kaggle/input", followlinks=True):
    path = Path(directory)
    if (path.name == "cifar-100-python" and
            required_cifar_files.issubset(filenames)):
        cifar_dirs.append(path.parent)
cifar_dirs = sorted(set(cifar_dirs), key=str)
if not cifar_dirs:
    visible = []
    for directory, dirnames, filenames in os.walk(
            "/kaggle/input", followlinks=True):
        visible.append({
            "directory": directory,
            "directories": sorted(dirnames)[:10],
            "files": sorted(filenames)[:10],
        })
        if len(visible) == 20:
            break
    raise FileNotFoundError(
        "Attach CIFAR-100 containing cifar-100-python/{train,test,meta}. "
        f"Visible input inventory: {visible}")
DATA_ROOT = cifar_dirs[0]
print("CIFAR-100 root:", DATA_ROOT)

if not (REPO / "experiments/vna_kaggle.py").exists():
    raise RuntimeError("Required module experiments/vna_kaggle.py is missing from the pinned commit")
for module in ("experiments/vna_provenance.py", "methods/vna.py"):
    if not (REPO / module).exists(): raise RuntimeError(f"Required module {module} is missing from the pinned commit")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)
sys.path[:0] = [str(REPO)]
print("Official TEST verified:", DATA_ROOT / "cifar-100-python/test")



In [ ]:
from concurrent.futures import ThreadPoolExecutor
from experiments.vna_kaggle import restore_stage_archives
from experiments.vna_kaggle import archive_outputs as _archive_outputs
STAGE_B = RUN_ROOT / "outputs"
STAGE_B.mkdir(parents=True, exist_ok=True)
ROOT=STAGE_B/RUN_TAG
ROOT.mkdir(parents=True,exist_ok=True)
restore_stage_archives(Path('/kaggle/input'), ROOT, RUN_TAG)
ENV = {**test_env, "PYTHONUNBUFFERED": "1", "TQDM_DISABLE": "1", "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2",
       "PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION": "python", "VNA_DEADLINE": str(VNA_DEADLINE)}
session_paused = threading.Event()
lane_state = {gpu: {"job": "idle", "last": ""} for gpu in range(2)}
def pause_due():
    if time.time() >= VNA_DEADLINE - 600: session_paused.set()
    return session_paused.is_set()
def launch(cmd, gpu, name, log_path):
    from experiments.vna_resume import launch_logged
    return launch_logged(cmd, cwd=REPO, env=gpu_env(gpu), log_path=log_path,
                         lane=gpu, job=name, lanes=lane_state,
                         paused=session_paused, deadline=VNA_DEADLINE)
def heartbeat():
    from experiments.vna_resume import print_heartbeat
    while not archive_stop.wait(600):
        print_heartbeat(RUN_ROOT / "outputs", SESSION_START, VNA_DEADLINE, lane_state)
archive_lock = threading.Lock(); archive_stop = threading.Event(); archive_errors = []
def archive_outputs():
    with archive_lock:
        return _archive_outputs(RUN_ROOT, budget=DISK_CAP_BYTES,
                                folders=("outputs/vna_stability_gap",), exclude_parts=("data",))
def periodic_archive():
    while not archive_stop.wait(3600):
        try: archive_outputs()
        except Exception as error:
            archive_errors.append(str(error)); print("Archive error:", error, flush=True)
def gpu_env(gpu): return {**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)}
archive_errors = []
lane_timings = {}
archiver = threading.Thread(target=periodic_archive, daemon=True); archiver.start()
heartbeat_thread = threading.Thread(target=heartbeat, daemon=True); heartbeat_thread.start()


In [ ]:
import queue
from concurrent.futures import ThreadPoolExecutor
from experiments.vna_resume import launch_logged
POLICY='schur_gamma_bound_rho25_v2'

def read(path,default=None):
    try:return json.loads(Path(path).read_text())
    except (FileNotFoundError,ValueError):return default

def checkpoint_epochs(directory):
    """epochs.json, or the full-precision hook.epochs saved in the run's own checkpoint (pre-epochs.json commits)."""
    epochs=read(directory/'epochs.json',[])
    if epochs:return epochs
    import torch
    for name in ('latest.pt','latest.previous.pt'):
        if (directory/name).exists():
            try:state=torch.load(directory/name,map_location='cpu',weights_only=False)
            except Exception as error:print('CHECKPOINT UNREADABLE',directory/name,error,flush=True);continue
            epochs=state.get('hook',{}).get('epochs',[])
            if epochs:return epochs
    return []

def staged_reference(job,directory):
    # Replays compare against protocol/tasks plus the recovered epoch losses of the same run.
    target=ROOT/'_references'/job
    target.mkdir(parents=True,exist_ok=True)
    for name in ('protocol.json','tasks.json','metric_trace_repair.json'):
        if (directory/name).exists():shutil.copy2(directory/name,target/name)
    (target/'epochs.json').write_text(json.dumps(checkpoint_epochs(directory)))
    (target/'source.txt').write_text(str(directory))
    return target

def p1_references():
    """Attached registered P1 jobs: noBN, superclass, lr scale 0.5, 20-task protocol, >= 5 tasks."""
    found={}
    for path in Path('/kaggle/input').rglob('protocol.json'):
        p=read(path,{})
        arm,seed=p.get('arm'),p.get('seed')
        if arm in ('icarl','icarl_full') and seed in (0,1) and p.get('backbone')=='resnet32_nobn':
            print(f'FOUND {arm}__s{seed} {path.parent}: commit={p.get("code_commit","")[:7]} order={p.get("class_order_kind")} '
                  f'lr={p.get("nobn_lr_scale")} solver={p.get("solver")} curvature={p.get("curvature")} '
                  f'max_tasks={p.get("comparison_protocol",{}).get("max_tasks")}',flush=True)
        if (p.get('setting')!='icarl' or p.get('algorithm_version')!='v4' or arm not in ('icarl','icarl_full')
                or seed not in (0,1) or p.get('backbone')!='resnet32_nobn' or p.get('class_order_kind')!='superclass'
                or p.get('nobn_lr_scale')!=.5 or 'max_tasks' in p.get('comparison_protocol',{})
                or p.get('fake_data') or p.get('preflight_only')):continue
        if arm=='icarl_full' and (p.get('solver'),p.get('curvature'),p.get('policy_version'))!=('gram','power10',POLICY):continue
        tasks=read(path.parent/'tasks.json',[]);epochs=checkpoint_epochs(path.parent)
        if len([t for t in tasks if t.get('task',99)<=5])<5 or len([e for e in epochs if e.get('task',99)<=5])<300:
            print(f'SKIP {arm}__s{seed} {path.parent}: tasks={len(tasks)} epochs={len(epochs)} commit={p.get("code_commit","")[:7]} '
                  f'status={read(path.parent/"summary.json",{}).get("status")}',flush=True)
            continue
        prefix=json.dumps(dict(e=[e['train_loss'] for e in epochs if e['task']<=5],t=[t for t in tasks if t['task']<=5]),sort_keys=True)
        found.setdefault(f'{arm}__s{seed}',{}).setdefault(prefix,path.parent)
    jobs={}
    for job in ('icarl_full__s0','icarl_full__s1','icarl__s0','icarl__s1'):
        candidates=list(found.get(job,{}).values())
        if not candidates:raise FileNotFoundError(f'Attach the P1 output containing {job} (>=5 tasks, epochs.json)')
        if len(candidates)>1:raise ValueError(f'{job}: attached P1 copies disagree on tasks 1-5: {candidates}')
        jobs[job]=candidates[0]
    commits={}
    for seed in (0,1):
        directory=jobs[f'icarl_full__s{seed}']
        schedule=read(directory/'protocol.json')['schedule']
        # A metric-trace repair relabels the run; its source_commit is the commit that ran it.
        # calibration_code_commit only names the K0 calibration source and is not used.
        repair=read(directory/'metric_trace_repair.json',{}).get('source_commit')
        commit=repair or schedule['code_commit']
        commits[f'icarl_full__s{seed}']=commits[f'icarl__s{seed}']=commit  # baseline replays on its VNA seed's source
    return jobs,commits

REFERENCES,COMMITS=p1_references()
REFERENCES={job:staged_reference(job,directory) for job,directory in REFERENCES.items()}
for job in ('icarl_full__s0','icarl__s0','icarl_full__s1','icarl__s1'):
    print(f'REPLAY {job}: commit {COMMITS[job]}  reference {(REFERENCES[job]/"source.txt").read_text()}',flush=True)
SOURCES={}
for commit in sorted(set(COMMITS.values())):
    source=Path('/kaggle/working')/f'p1_source_{commit[:12]}'
    if not (source/'.git').exists():
        private_clone(MAIN_URL,source,commit)
        subprocess.run(['bash','scripts/fetch_third_party.sh'],cwd=source,check=True)
    head=subprocess.check_output(['git','rev-parse','HEAD'],cwd=source,text=True).strip()
    if head!=commit:raise RuntimeError(f'checkout {source} is {head}, expected {commit}')
    SOURCES[commit]=source
errors=[]

def replay(gpu,job):
    out=ROOT/job
    if read(out/'stability_gap'/'report.json',{}).get('status')=='OK':return True
    source=SOURCES[COMMITS[job]]
    env={**gpu_env(gpu),'PYTHONPATH':str(source)}
    cmd=[sys.executable,'-u',str(REPO/'diagnostics'/'stability_gap.py'),'--repo',str(source),'--commit',COMMITS[job],
         '--reference',str(REFERENCES[job]),'--job',job,'--out',str(out),'--data-dir',str(DATA_ROOT)]
    return launch_logged(cmd,cwd=source,env=env,log_path=RUN_ROOT/f'{job}.log',lane=gpu,job=job,
                         lanes=lane_state,paused=session_paused,deadline=VNA_DEADLINE)

def groupnorm(gpu,seed):
    out=ROOT/f'groupnorm__s{seed}'
    if (out/'gn_vo_h_summary.json').exists():return True
    cmd=[sys.executable,'-u','-m','diagnostics.gn_vo_h','--seed',str(seed),'--out',str(out),'--data-dir',str(DATA_ROOT)]
    return launch(cmd,gpu,f'groupnorm__s{seed}',RUN_ROOT/f'groupnorm__s{seed}.log')

def lane(gpu,seed):
    for name,work in ((f'icarl_full__s{seed}',lambda:replay(gpu,f'icarl_full__s{seed}')),
                      (f'icarl__s{seed}',lambda:replay(gpu,f'icarl__s{seed}')),
                      (f'groupnorm__s{seed}',lambda:groupnorm(gpu,seed))):
        if pause_due():return
        try:
            if not work():
                if pause_due():return
                errors.append(dict(job=name,error='non-zero exit; see log'))
        except Exception as error:errors.append(dict(job=name,error=f'{type(error).__name__}: {error}'))
        archive_outputs()

try:
    with ThreadPoolExecutor(max_workers=2) as pool:
        for job in [pool.submit(lane,0,0),pool.submit(lane,1,1)]:job.result()
finally:
    archive_stop.set();archiver.join(timeout=10);archive_outputs()
if session_paused.is_set():print('PAUSED: attach this output and rerun to continue.')
if errors:print('JOB ERRORS:',json.dumps(errors,indent=1),flush=True)


In [ ]:
import matplotlib.pyplot as plt
# GroupNorm first: it does not depend on the P1 replays and is printed even if a replay mismatches.
for seed in (0,1):
    summary=read(ROOT/f'groupnorm__s{seed}'/'gn_vo_h_summary.json',{})
    print('GROUPNORM',seed,json.dumps(summary,indent=1) if summary else 'MISSING')
reports={job:read(ROOT/job/'stability_gap'/'report.json',{}) for job in ('icarl__s0','icarl_full__s0','icarl__s1','icarl_full__s1')}
bad={job:(r.get('status') or 'MISSING') for job,r in reports.items() if r.get('status')!='OK'}
if bad:
    for job in bad:print(job,'MATCH:',json.dumps(read(ROOT/job/'stability_gap'/'match.json',{}),indent=1)[:3000])
    raise RuntimeError(f'Stability-gap measurements withheld until every replay matches P1 exactly: {bad}')
rows=[]
for job,r in reports.items():
    for t in r['tasks']:
        rows.append(dict(job=job,task=t['task'],pre_switch_old=t['pre_switch_old'],r0_old=t['r0_old'],min_old=t['min_old'],
            min_step=t['min_old_step'],gap_depth=t['gap_depth'],recovery_to_pre=t['recovery_step_to_pre_switch'],
            recovery_to_half=t['recovery_step_to_half_gap'],end_old=t['end_old'],end_new=t['end_new']))
print('STABILITY GAP (old-class CNN accuracy, %; steps = SGD updates since the task switch):')
for row in rows:print(json.dumps(row))
for seed in (0,1):
    for t in reports[f'icarl_full__s{seed}']['tasks']:
        for e in t['interventions']:print(json.dumps(dict(seed=seed,task=t['task'],**e)))
fig,axes=plt.subplots(2,4,figsize=(20,8),sharey=True)
for i,seed in enumerate((0,1)):
    for j,task in enumerate((2,3,4,5)):
        ax=axes[i][j]
        for arm,style in (('icarl','-'),('icarl_full','--')):
            t=next(x for x in reports[f'{arm}__s{seed}']['tasks'] if x['task']==task)
            pts=[p for p in t['curve'] if p['r']<=400]
            ax.plot([p['r'] for p in pts],[p['old'] for p in pts],style,label=arm)
            if arm=='icarl_full':
                for e in t['interventions']:
                    if e['step']<=400:ax.axvline(e['step'],color='grey',alpha=.3)
            ax.axhline(t['pre_switch_old'],color='k',lw=.5)
        ax.set(title=f'seed {seed} task {task}',xlabel='step in task',ylabel='old-class acc')
        ax.legend(fontsize=7)
fig.tight_layout();fig.savefig(ROOT/'stability_gap.png',dpi=120);plt.show()
